# Explanation consistency: SHAP-like attributions, LIME-like surrogates and alignment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/explainability.ipynb)

Companion notebook for the ReliableML paper `docs/latex/explainability.tex`. It uses `explain_estimator` (or `ReliablePipeline.explain`): permutation-sampled Shapley attributions, local ridge surrogates, and rank/sign/top-feature agreement between them.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Quantities

For a model output $f$ (here the predicted probability of the explained class), $d$ features $F$ and a value function $v_x(S)$ that sets features in $S$ to the instance's values and the rest to background draws, the Shapley value is
$$\phi_j(f,x)=\sum_{S\subseteq F\setminus\{j\}}\frac{|S|!\,(d-|S|-1)!}{d!}\,\bigl[v_x(S\cup\{j\})-v_x(S)\bigr].$$
ReliableML *estimates* it by Monte-Carlo permutation sampling (`num_samples` permutations, so results carry sampling error and depend on the background data). With a single fixed baseline the efficiency property is $\sum_j\phi_j=f(x)-f(\text{baseline})$.

**LIME-like surrogate.** Perturbations $z$ are drawn from marginal Gaussians built from the background, weighted by a distance kernel $\pi_x(z)$, and a ridge-stabilized weighted linear model is fitted:
$$g_x^\star=\arg\min_{g\in\mathcal G}\sum_{z\in\mathcal Z}\pi_x(z)\bigl(f(z)-g(z)\bigr)^2+\Omega(g),\qquad \pi_x(z)=\exp\!\bigl(-(\lVert z-x\rVert/\sigma)^2\bigr).$$
The weighted local $R^2$ ("fidelity") says how well the surrogate fits its *synthetic* neighbourhood.

**Global importance and alignment.** $I_j=\frac1m\sum_{i=1}^m\lvert\hat\phi_j(f,x_i)\rvert$. Between the SHAP-like and LIME-like vectors the report gives per-row Spearman rank correlation of absolute importances, top-feature overlap and sign agreement, and a bootstrap interval for the mean Spearman. These measure *agreement between two explanation procedures*, not correctness of the explanation or evidence of causation.

## 2. Breast cancer: global and local explanations

In [ ]:
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml_reliability import explain_estimator

warnings.filterwarnings("ignore")
bc = load_breast_cancer(as_frame=True)
features = ["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean concavity", "mean symmetry"]
X, y = bc.data[features], bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = RandomForestClassifier(n_estimators=150, random_state=0).fit(Xtr, ytr)

report = explain_estimator(model, Xte, y_true=yte, background_data=Xtr.sample(100, random_state=0), class_label=1,
                           max_instances=30, num_samples=300, random_seed=0, bootstrap_samples=200)
global_importance = pd.DataFrame(report.global_shap_importance)
print(global_importance.round(4).to_string(index=False))
print("alignment:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in report.reason_alignment.items()})
print("warnings:", report.warnings)

In [ ]:
local = report.local_explanations[0]
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in local.items()
       if k in ["row_index", "prediction", "predicted_probability_for_class", "lime_local_fidelity_r_squared",
                "absolute_importance_spearman", "top_feature_overlap", "sign_agreement"]})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
global_importance.sort_values("mean_absolute_shapley_value").plot.barh(x="feature", y="mean_absolute_shapley_value", ax=axes[0], legend=False)
axes[0].set_title("Global: mean |SHAP|")
compare = pd.DataFrame({"SHAP": local["shap_values"], "LIME": local["lime_weights"]})
compare.plot.barh(ax=axes[1]); axes[1].set_title(f"Local explanation, row {local['row_index']}"); axes[1].axvline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

### Efficiency check

The Shapley values of one instance should add up to $f(x)$ minus the average prediction over the background (approximately, because background rows and permutations are sampled).

In [ ]:
background = Xtr.sample(100, random_state=0)
mean_background = model.predict_proba(background)[:, list(model.classes_).index(1)].mean()
rows = []
for local in report.local_explanations[:8]:
    total = sum(local["shap_values"].values())
    rows.append({"row": local["row_index"], "f(x)": local["predicted_probability_for_class"],
                 "f(x) - mean f(background)": local["predicted_probability_for_class"] - mean_background, "sum of SHAP": total})
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 3. Synthetic model with known ground truth

A logistic data-generating process with coefficients $(2,-1,0.5,0,0)$ on standardized features lets us compare the recovered global ranking with the truth.

In [ ]:
rng = np.random.default_rng(5)
n = 3000
coef = np.array([2.0, -1.0, 0.5, 0.0, 0.0])
Xs = pd.DataFrame(rng.normal(size=(n, 5)), columns=[f"x{i}" for i in range(5)])
ys = (rng.random(n) < 1 / (1 + np.exp(-(Xs.to_numpy() @ coef)))).astype(int)
Xstr, Xste, ystr, yste = train_test_split(Xs, ys, test_size=0.3, random_state=0)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xstr, ystr)

true_importance = dict(zip(Xs.columns, np.abs(coef)))
rep_s = explain_estimator(lr, Xste, background_data=Xstr.sample(200, random_state=0), class_label=1, max_instances=40,
                          num_samples=300, random_seed=0, bootstrap_samples=200, feature_importance=true_importance)
imp = pd.DataFrame(rep_s.global_shap_importance).set_index("feature")["mean_absolute_shapley_value"]
table = pd.DataFrame({"mean |SHAP|": imp, "true |coef|": pd.Series(true_importance)})
print(table.round(3).to_string())
print("Spearman(mean |SHAP|, true |coef|) =", round(stats.spearmanr(table["mean |SHAP|"], table["true |coef|"])[0], 3))
sia = rep_s.supplied_importance_alignment
print("supplied-importance alignment: global Spearman =", round(sia["global_absolute_importance_spearman"], 3),
      "| mean per-row top-feature overlap =", round(np.mean([r["top_feature_overlap"] for r in sia["per_row"]]), 3))

### Sensitivity to `num_samples` and the random seed

In [ ]:
rows = []
for num_samples in (100, 300, 1000):
    for seed in (0, 1):
        r = explain_estimator(lr, Xste, background_data=Xstr.sample(200, random_state=0), class_label=1, max_instances=20,
                              num_samples=num_samples, random_seed=seed, bootstrap_samples=100)
        top = pd.DataFrame(r.global_shap_importance).set_index("feature")["mean_absolute_shapley_value"]
        rows.append({"num_samples": num_samples, "seed": seed, "mean |SHAP| x0": top["x0"], "x1": top["x1"], "x3 (null)": top["x3"],
                     "alignment": r.reason_alignment["alignment_score_0_to_1"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 4. Optional export (privacy caution)

`export_path` writes the **input rows, predictions, optional labels and top features** to disk - that can contain personal or sensitive data, so it is opt-in and requires covering every row.

In [ ]:
small = Xte.iloc[:10]
explain_estimator(model, small, y_true=yte.iloc[:10], background_data=Xtr.sample(50, random_state=0), class_label=1,
                  num_samples=100, random_seed=0, bootstrap_samples=100, export_path="explanations_demo.csv")
print(pd.read_csv("explanations_demo.csv").iloc[:3, :8].round(3).to_string(index=False))
os.remove("explanations_demo.csv")  # do not leave row-level data lying around

## Interpretation and limitations

**How to read the output.** Global importance ranks features by average absolute attribution; local views show how one prediction is assembled. The alignment numbers say whether two different explanation procedures tell a similar story, and the sensitivity table shows how much of the output is just sampling noise from `num_samples` and the seed. In the synthetic case, the null features should receive small attributions.

**Limitations.** Attributions describe the model's behaviour under a chosen background distribution and perturbation scheme; they are not causal effects. Correlated or constrained features can produce unrealistic perturbed rows; Monte-Carlo estimates are noisy at small `num_samples`; the LIME-like fidelity only measures fit to a synthetic neighbourhood; agreement between the two methods is not correctness; only finite numeric features and models with `predict_proba` are supported; exported files can leak sensitive data.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.